# Day 28: Prefix Cache：先正确再优化

选择精确完整前缀缓存；命中时复用 K/V 与最后 logits，克隆以隔离后续写入。不做 copy-on-write，不承诺节省内存；权重变动后必须清空 store。

参考源码：`learning/mini_vllm/engine.py`。

每日概念、误区、面试问答：[知识讲义](../docs/KNOWLEDGE_DAY22_30.md)。

当前个人进度以[进度记录](../docs/LEARNING_PROGRESS.md)为准；运行参考答案不等于已独立掌握。

[Day 28完整讲义](../docs/DAY28_PREFIX_CACHE.md)：概念、计算契约、源码实验、调试优化、答案与资料。

先写自己的预测，再执行原有代码单元格；不要自动勾选个人验收。

## 1. 图解与数字例子

公共系统提示像一份已计算的草稿，不必每次从头算：

```text
前缀[1,2,3] → 首次prefill → 缓存KV与末位置logits
再次[1,2,3] → hit → 拿一份隔离副本继续输入4
原前缀仍长3，当前分支变长4
```

本实验只识别**完整精确token前缀**。
`[1,2,3,4]`不会自动查找已存的`[1,2,3]`最长部分命中。
“文字意思相近”不是token前缀相同；不同tokenizer也不能直接共享。

## 先预测

执行前写下预期 shape、输出或调度顺序；在下面另建 Markdown 单元格保存自己的答案，不修改参考实现作为唯一证据。

In [ ]:
import json
import os
import sys
from pathlib import Path
root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/curriculum.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import torch
from learning.curriculum import run_day
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


In [ ]:
report = run_day(28, DEVICE)
print(json.dumps(report, indent=2, ensure_ascii=False))
destination = Path('logs') / f"day{report['day']:02d}-course.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(report, indent=2, ensure_ascii=False) + '\n')
print('Report:', destination)


## 自测与失败实验

只缓存前缀最后 logits、不缓存 K/V，能继续 decode 吗？

记录你改变的唯一变量、预期、实际结果和差异原因。

`requires_cuda` / `requires_cuda_compiler` / `external_service_not_configured` 都不是通过。

## 5. 自测与参考答案

**问：为什么只缓存logits不够？**
答：继续生成时新Q还要读历史KV，logits不能重建那些状态。

**问：clone为什么可能不省内存？**
答：它复制快照与返回状态，优先保证隔离，不等于物理页共享。

**问：权重变了仍能命中相同token key吗？**
答：当前代码可能照样命中，所以必须主动clear；生产应做更完整身份与失效策略。

**问：是否实现自动最长前缀匹配？**
答：没有，本例只匹配完整精确前缀。

完整实现限制、调试步骤与资料见[本日讲义](../docs/DAY28_PREFIX_CACHE.md)。

以上是参考答案，不是个人学习结果。

## 验收记录

- [ ] 能解释原理
- [ ] 能自行复现
- [ ] 能解释一次失败
- [ ] 能说明当前实现不支持什么

个人笔记：

待填写。